# 1、使用FUNCTION_CALL模式

In [4]:
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.agents import AgentType, initialize_agent, create_tool_calling_agent, AgentExecutor
from langchain_classic.tools import Tool
from langchain_community.tools import TavilySearchResults
from langchain_openai import ChatOpenAI
import os
import dotenv
dotenv.load_dotenv()

# 读取配置文件
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

# 获取Tavily搜索工具实例
search = TavilySearchResults(max_results = 3)
# 2、获取对应工具实例

search_tool = Tool(
    func = search.run,
    name = "search",
    description = "用于检索互联网上的信息",
)
# 3、获取大模型
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)
memory = ConversationBufferMemory(
    return_messages=True,
    memory_key="chat_history",
)
# 4、提示词模版（ChatPromptTemplate为例）
prompt_template = ChatPromptTemplate.from_messages([
    ("system","你是一个乐于助人的ai助手，根据用户的提问，必要时调用search工具，使用互联网检索数据"),
    ("system","{chat_history}"), # 添加一个chat_history的变量，用于记录上下文记忆
    ("human","{input}"),
    ("system","{agent_scratchpad}"), # agent_scratchpad必须声明  agent_scratchpad 放在 input 之前确实可能导致输出重复或混乱
])

# 5、获取Agent实例：create_tool_calling_agent()
agent = create_tool_calling_agent(
    llm = llm,
    prompt = prompt_template,
    tools = [search_tool],
)
# 6、获取AgentExecutor实例
agent_executor = AgentExecutor(
    agent = agent,
    tools = [search_tool],
    verbose = True,
    memory = memory,
)
# 7、通过AgentExecutor实例调用invoke方法，得到响应
response = agent_executor.invoke({"input":"查询今天的西安天气情况"})
# 8、处理响应
print(response)



> Entering new AgentExecutor chain...

Invoking: `search` with `西安天气情况 2023年10月5日`


[{'title': '西安市10月份气温查询- 西安历史天气 - 天气二四网', 'url': 'https://www.tianqi24.com/xian/history10.html', 'content': '# 西安市10月份气温西安历史天气\n\n## 西安10月份气温怎样？\n\n西安市十月份气温，根据2025年、2024年、2023年中每年的10月的历史天气数据统计分析，共有：\n\n平均高温：21℃， 最高温度(2024年10月15日)： 29℃，\n\n平均低温：11℃； 最低温度(2025年10月21日)： 3℃；\n\n平均空气综合质量AQI值是： 42 (优)\n\n空气最好：10 优， 空气最差：85 良；\n\n西安近3年的每年10月份：月均降雨（8）天, 月均降雨量（188.87）毫米！\n\n  \n\n### 西安2025年、2024年、2023年10月天气分布图\n\n  \n  \n\n### 西安10月温度走势图\n\n### 西安市10月份气温： 2023年 ~ 2025年\n\n 雨12天  \n  晴8天  \n  阴6天  \n  多云5天\n\n  多云10天  \n  晴9天  \n  雨7天  \n  阴5天\n\n  晴11天  \n  阴7天  \n  多云7天  \n  雨5天  \n  沙雾霾1天\n 11℃\n\n  10℃\n 月降雨量\n\n  479.3 mm\n\n  0.3 mm\n\n  87 mm\n 日均PM25\n\n  25 优\n\n  40 优\n\n  29 优\n 日均最高风速\n\n  7.8 Km/h\n\n  8.3 Km/h\n\n  5.6 Km/h\n 该月最高温度\n\n  25℃  \n  10月08日\n\n  29℃  \n  10月15日\n\n  26℃  \n  10月15日\n 该月最低温度\n\n  7℃  \n  10月21日\n\n  3℃  \n  10月23日\n\n  6℃  \n  10月21日\n\n### 西安市2025年10月份历史天气\n\n 日期\n\

In [6]:
response = agent_executor.invoke({"input":"上海呢？"})
print(response)



> Entering new AgentExecutor chain...

Invoking: `search` with `上海天气 2023年10月5日`


[{'title': '2023年10月5日上海已入秋(附历年入秋时间) - 上海本地宝', 'url': 'http://sh.bendibao.com/news/2023109/278164.shtm', 'content': '民生\n 办事\n 社保\n 公积金\n 交通\n 地铁\n 旅游\n 景点\n 活动\n 教育\n 网点\n 招聘\n 房产\n 特惠\n\n 生活指南\n 关注\n 生活\n 经济\n 旅游\n 社会\n 教育\n 文体\n 提醒\n 新政\n\n上海本地宝 > 资讯快递 > 上海天气 > 2023年10月5日上海已入秋(附历年入秋时间)\n\n2023-10-09 17:36 【我要纠错】\n\n 下载指南\n\n分享至\n\n【导语】：上海市气象局官方微博消息：2023年10月5日上海入秋。上海已经进入气象意义上的秋天了！\n\n2023上海入秋时间\n\n\u3000\u3000▪ 入秋时间：10月5日\n\n\u3000\u3000▪ 入秋标准：连续5天日平均气温小于22℃\n\n\u3000\u3000自2023年10月5日起，徐家汇站已经连续4天日平均气温小于22℃。10月9日02时、08时、14时的气温分别为18.9℃、19.7℃和21.9℃，也就是说，20时的气温低于27.5℃即能符合入秋标准;也就是说，上海10月5日已经进入气象意义上的秋天了。\n\n附：上海历年入秋时间\n\n上海近15年入秋时间\n\n温馨提示：微信搜索公众号【上海本地宝】，关注后在对话框回复【入秋】可查询上海历年入秋时间、上海入秋标准。\n\n手机访问 上海本地宝首页\n\n相关推荐 上海入秋时间 上海入秋标准\n\n- 2024年10月2日上海已入秋(附历年入秋时间）\n\n  10月2日至今，已经连续5天日平均气温低于22℃，这也就是说上海气象意义上的秋天已经来到！10月2日即为今年的入秋日。\n- 上海入秋时间一览表(历年汇总）\n\n  2023上海入秋时间：10月5日；2022上海入秋时间：10月5日；2021上海入秋时间：10月16日，近15年入秋时间如下。\n-

# 2、使用ReAct模式

示例1：使用网络上现成的提示词模版结构

In [7]:
from langchain_classic import hub
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_classic.agents import AgentType, initialize_agent, create_tool_calling_agent, AgentExecutor, create_react_agent
from langchain_core.tools import StructuredTool
from langchain_classic.tools import Tool
from langchain_community.tools import TavilySearchResults
from langchain_openai import ChatOpenAI
import os
import dotenv
dotenv.load_dotenv()

# 读取配置文件
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

# 获取Tavily搜索工具实例
search = TavilySearchResults(max_results = 3)
# 2、获取对应工具实例

search_tool = Tool(
    func = search.run,
    name = "search",
    description = "用于检索互联网上的信息",
)
# 3、获取大模型
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)
# 4、使用LangChain Hub中的官方ReAct提示模板
prompt_template = hub.pull("hwchase17/react-chat")

memory = ConversationBufferMemory(
    return_messages= True,
    memory_key="chat_history",
)

# 5、获取Agent实例：create_react_agent()
agent = create_react_agent(
    llm = llm,
    prompt = prompt_template,
    tools = [search_tool],
)
# 6、获取AgentExecutor实例
agent_executor = AgentExecutor(
    agent = agent,
    tools = [search_tool],
    verbose = True,
    memory = memory,
)
# 7、通过AgentExecutor实例调用invoke方法，得到响应
response = agent_executor.invoke({"input":"查询今天的西安天气情况"})
# 8、处理响应
print(response)



> Entering new AgentExecutor chain...
```
Thought: Do I need to use a tool? Yes
Action: search
Action Input: "西安天气情况 2023年10月25日"
```[{'title': '西安市在十月2026 的天氣：氣溫與氣候 - Climate-Data.org', 'url': 'https://zh.climate-data.org/%E4%BA%9E%E6%B4%B2/%E4%B8%AD%E5%9B%BD/%E9%99%95%E8%A5%BF/%E8%A5%BF%E5%AE%89%E5%B8%82-2613/t/%E5%8D%81%E6%9C%88-10/', 'content': 'Partner-Links\n预订您的酒店：\n西安市\n预订您的度假公寓：\n西安市\n飞往以下城市的特价航班：\n西安市\n\n## 十月 西安市 的气温\n\n氣候圖 // 每月天氣, 西安市\n\n十月 在 西安市 中最熱的一天是 02.10，當天最高溫為 21.9°C，而最冷的一天是 31.10，最低溫降至 6.6°C。 月初的日最高平均溫度為 20.8°C，最低平均溫度為 12.2°C。 月中時，日最高平均溫度約為 19.1°C，最低平均溫為 10.2°C。 到月底，平均最高溫約為 17.3°C，最低溫接近 7.8°C。 最熱的連續 10 天期間是 1-10，日最高溫約為 16.3°C。 最冷的連續 10 天區間是 21-31，平均最低溫為 12.4°C。\n\n十月 在 西安市 的日照時間平均介於每天 7 小時至 9 小時之間。最明亮的一天 07.10 大約有 9 小時，而日照最少的 24.10 則只有 7 小時。月初三分之一期間的日照平均為 8 小時；第二個三分之一為 8 小時；月底三分之一為 8 小時。最晴朗的 10 天階段是從 1-10，每天約有 8 小時日照。最陰天的階段是 21-31，只有 8 小時。\n\n在 十月 ，西安市 的降水量介於每日總量 2.7 毫米/英吋與 14.3 毫米/英吋之間波動。最濕潤的一天是 30.10，降水量為 14.3 毫米/英吋，最乾燥的則是 01.10，僅有 2.7 毫米/英吋。前十天的平均降雨量為 5.6 毫米/英

In [8]:
response = agent_executor.invoke({"input":"西安2026年3月30日的呢"})
print(response)



> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes  
Action: search  
Action Input: "西安 2026年3月30日 天气预报"  [{'title': '西安市气象台2026年3月9日16时发布“春运”专题天气预报 - QQ', 'url': 'https://news.qq.com/rain/a/20260309A06H7800', 'content': '# 西安市气象台2026年3月9日16时发布“春运”专题天气预报\n\n头像\n\n西安发布\n\n2026-03-09 16:50发布于陕西西安发布官方账号\n\n【西安市气象局】西安市气象台2026年3月9日16时发布“春运”专题天气预报：预计未来三天我市以晴到多云天气为主，最高气温17～20℃。13-14日我市有一次雨雪、降温及吹风天气，主要降水时段在13日白天至14日白天，全市以小到中雨为主，南部高海拔山区雨夹雪转雪，最高气温下降至10℃左右。\n\n具体预报如下：\n\n今天晚上：晴间多云，4℃～10℃；\n\n10日：晴间多云，6℃～18℃；\n\n11日：晴转多云，7℃～20℃，阵风5-6级；\n\n12日：多云转阴，7℃～17℃，阵风5-6级；\n\n13日：小到中雨，南部高海拔山区雨夹雪转雪，7℃～11℃；\n\n14日：小雨转阴天，南部山区小雪转阴天，3℃～10℃；\n\n15日：阴天，2℃～11℃；\n\n16日：多云，3℃～13℃。\n\n责编：王晨', 'score': 0.9995308}, {'title': '西安市气象台2026年03月15日16时发布天气预报 - 搜狐', 'url': 'https://m.sohu.com/a/996799874_121443915?scm=10001.325_13-325_13.0.0-0-0-0-0.5_1334', 'content': '## 西安市气象台2026年03月15日16时发布天气预报\n\n【西安市气象局】西安市气象台2026年03月15日16时发布天气预报：明天白天我市以多云天气为主，最高气温升至15℃；明天夜间至17日白天我市将迎来一次降水、降温天气过程，城区及平原地区以小雨为主，南

示例2：自定义提示词模版

In [9]:
from langchain_classic import hub
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_classic.agents import AgentType, initialize_agent, create_tool_calling_agent, AgentExecutor, create_react_agent
from langchain_core.tools import StructuredTool
from langchain_classic.tools import Tool
from langchain_community.tools import TavilySearchResults
from langchain_openai import ChatOpenAI
import os
import dotenv
dotenv.load_dotenv()

# 读取配置文件
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

# 获取Tavily搜索工具实例
search = TavilySearchResults(max_results = 3)
# 2、获取对应工具实例

search_tool = Tool(
    func = search.run,
    name = "search",
    description = "用于检索互联网上的信息",
)
# 3、获取大模型
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)
# 4、使用自定义提示词模版
template = """
Assistant is a large language model trained by OpenAI.

Assistant is designed to be able to assist with a wide range of tasks, from answering simple questions to providing in-depth explanations and discussions on a wide range of topics. As a language model, Assistant is able to generate human-like text based on the input it receives, allowing it to engage in natural-sounding conversations and provide responses that are coherent and relevant to the topic at hand.

Assistant is constantly learning and improving, and its capabilities are constantly evolving. It is able to process and understand large amounts of text, and can use this knowledge to provide accurate and informative responses to a wide range of questions. Additionally, Assistant is able to generate its own text based on the input it receives, allowing it to engage in discussions and provide explanations and descriptions on a wide range of topics.

Overall, Assistant is a powerful tool that can help with a wide range of tasks and provide valuable insights and information on a wide range of topics. Whether you need help with a specific question or just want to have a conversation about a particular topic, Assistant is here to assist.

TOOLS:
------

Assistant has access to the following tools:

{tools}

To use a tool, please use the following format:

```
Thought: Do I need to use a tool? Yes
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
```

When you have a response to say to the Human, or if you do not need to use a tool, you MUST use the format:

```
Thought: Do I need to use a tool? No
Final Answer: [your response here]
```

Begin!

Previous conversation history:
{chat_history}

New input: {input}
{agent_scratchpad}
"""
prompt_template = PromptTemplate.from_template(
    template = template,
)

memory = ConversationBufferMemory(
    return_messages= True,
    memory_key="chat_history",
)

# 5、获取Agent实例：create_react_agent()
agent = create_react_agent(
    llm = llm,
    prompt = prompt_template,
    tools = [search_tool],
)
# 6、获取AgentExecutor实例
agent_executor = AgentExecutor(
    agent = agent,
    tools = [search_tool],
    verbose = True,
    memory = memory,
)
# 7、通过AgentExecutor实例调用invoke方法，得到响应
response = agent_executor.invoke({"input":"查询今天的西安天气情况"})
# 8、处理响应
print(response)



> Entering new AgentExecutor chain...
```
Thought: Do I need to use a tool? Yes
Action: search
Action Input: "西安天气情况 2023年10月25日"
```[{'title': '西安市在十月2026 的天氣：氣溫與氣候 - Climate-Data.org', 'url': 'https://zh.climate-data.org/%E4%BA%9E%E6%B4%B2/%E4%B8%AD%E5%9B%BD/%E9%99%95%E8%A5%BF/%E8%A5%BF%E5%AE%89%E5%B8%82-2613/t/%E5%8D%81%E6%9C%88-10/', 'content': 'Partner-Links\n预订您的酒店：\n西安市\n预订您的度假公寓：\n西安市\n飞往以下城市的特价航班：\n西安市\n\n## 十月 西安市 的气温\n\n氣候圖 // 每月天氣, 西安市\n\n十月 在 西安市 中最熱的一天是 02.10，當天最高溫為 21.9°C，而最冷的一天是 31.10，最低溫降至 6.6°C。 月初的日最高平均溫度為 20.8°C，最低平均溫度為 12.2°C。 月中時，日最高平均溫度約為 19.1°C，最低平均溫為 10.2°C。 到月底，平均最高溫約為 17.3°C，最低溫接近 7.8°C。 最熱的連續 10 天期間是 1-10，日最高溫約為 16.3°C。 最冷的連續 10 天區間是 21-31，平均最低溫為 12.4°C。\n\n十月 在 西安市 的日照時間平均介於每天 7 小時至 9 小時之間。最明亮的一天 07.10 大約有 9 小時，而日照最少的 24.10 則只有 7 小時。月初三分之一期間的日照平均為 8 小時；第二個三分之一為 8 小時；月底三分之一為 8 小時。最晴朗的 10 天階段是從 1-10，每天約有 8 小時日照。最陰天的階段是 21-31，只有 8 小時。\n\n在 十月 ，西安市 的降水量介於每日總量 2.7 毫米/英吋與 14.3 毫米/英吋之間波動。最濕潤的一天是 30.10，降水量為 14.3 毫米/英吋，最乾燥的則是 01.10，僅有 2.7 毫米/英吋。前十天的平均降雨量為 5.6 毫米/英

In [ ]:
response = agent_executor.invoke({"input":"西安2026年3月30日的呢"})
print(response)